# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Brandon Acosta

**Date:** May 12, 2026

**Option:** B — Job Fit Analyzer  

**API Path:** Paid

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [6]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("OpenAI API key loaded successfully.")
else:
    print("OpenAI API key not found. Check your .env file.")

OpenAI API key loaded successfully.


In [23]:
# ── Install packages (uncomment as needed) ──
#!pip install langchain-core
#!pip install langchain_openai
#!pip install langchain-text-splitters
#!pip install langchain-chroma
#!pip install langchain_community

# ── Load API keys from .env ──
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# ── Your imports below ──
from pathlib import Path
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

try: 
    from langchain_chroma import Chroma 
except ImportError: 
    from langchain_community.vectorstores import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [27]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=openai_api_key
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=openai_api_key
)

test_vector = embeddings.embed_query("Testing the embedding model for the Job Fit Analyzer.")

print("Embedding model loaded successfully.")
print("Vector length:", len(test_vector))

test_response = llm.invoke("Reply with exactly: model connection successful")

print(test_response.content)

Embedding model loaded successfully.
Vector length: 1536
model connection successful


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [31]:
# ── Load JD metadata ──

from pathlib import Path
import pandas as pd

# Define project paths
DATA_DIR = Path("../data")
JD_DIR = DATA_DIR / "job_descriptions"
RESUME_DIR = DATA_DIR / "resume"
METADATA_PATH = DATA_DIR / "jd_metadata.csv"

# Load metadata CSV
jd_metadata = pd.read_csv(METADATA_PATH)

# Preview metadata
print("JD metadata loaded successfully.")
print("Number of rows in metadata:", len(jd_metadata))

display(jd_metadata.head(10))

JD metadata loaded successfully.
Number of rows in metadata: 10


,filename,company,title,source_url,date_collected
0,jd_01_sales_operations_analyst.txt,JM Eagle,Sales Operations Analyst,https://www.linkedin.com/jobs/view/4405059025,5/12/26
1,jd_02_revenue_operations_associate.txt,Sunbit,Revenue Operations Associate,https://www.linkedin.com/jobs/view/4406226503,5/12/26
2,jd_03_crm_operations_associate.txt,FanDuel,CRM Operations Associate,https://www.linkedin.com/jobs/view/4361065784,5/12/26
3,jd_04_gtm_operations_manager.txt,Cast & Crew,GTM Operations Manager,https://www.linkedin.com/jobs/view/4407973056,5/12/26
4,jd_05_business_analyst.txt,Salted,Business Analyst,https://www.linkedin.com/jobs/view/4387786174,5/12/26
5,jd_06_data_analyst.txt,Skechers,Operations Data & Financial Analyst,https://www.linkedin.com/jobs/view/4408048076,5/12/26
6,jd_07_business_intelligence_analyst.txt,Greater Good Health,Business Intelligence Analyst,https://www.linkedin.com/jobs/view/4410229599,5/12/26
7,jd_08_operations_analyst.txt,Masimo,Operations Analyst,https://www.linkedin.com/jobs/view/4378611641,5/12/26
8,jd_09_strategy_analyst.txt,Curana Health,"Business Analyst, Strategic Operations",https://www.linkedin.com/jobs/view/4407798544,5/12/26
9,jd_10_associate_consultant_operations.txt,O Positiv Health,Associate Strategy Manager,https://www.linkedin.com/jobs/view/4399497055,5/12/26


In [33]:
# ── Load JD documents and resume ──

jd_documents = []
resume_documents = []

# Load job description text files
for _, row in jd_metadata.iterrows():
    filename = row["filename"]
    file_path = JD_DIR / filename
    
    if file_path.exists():
        text = file_path.read_text(encoding="utf-8")
        
        doc = Document(
            page_content=text,
            metadata={
                "doc_type": "job_description",
                "filename": filename,
                "company": row["company"],
                "title": row["title"],
                "source_url": row["source_url"],
                "date_collected": row["date_collected"]
            }
        )
        
        jd_documents.append(doc)
    
    else:
        print(f"Missing file: {file_path}")

# Load resume text file
resume_files = list(RESUME_DIR.glob("*.txt"))

if len(resume_files) == 0:
    print("No resume .txt file found in data/resume/")
else:
    resume_path = resume_files[0]
    resume_text = resume_path.read_text(encoding="utf-8")
    
    resume_doc = Document(
        page_content=resume_text,
        metadata={
            "doc_type": "resume",
            "filename": resume_path.name
        }
    )
    
    resume_documents.append(resume_doc)

print("Job descriptions loaded:", len(jd_documents))
print("Resume documents loaded:", len(resume_documents))


Job descriptions loaded: 10
Resume documents loaded: 1


In [35]:
# ── Preview sample content ──

print("=== Sample Job Description ===")
print("Company:", jd_documents[0].metadata["company"])
print("Title:", jd_documents[0].metadata["title"])
print("Filename:", jd_documents[0].metadata["filename"])
print()
print(jd_documents[0].page_content[:1000])

print("\n" + "=" * 80 + "\n")

print("=== Resume Preview ===")
print("Filename:", resume_documents[0].metadata["filename"])
print()
print(resume_documents[0].page_content[:1000])

=== Sample Job Description ===
Company: JM Eagle
Title: Sales Operations Analyst
Filename: jd_01_sales_operations_analyst.txt

Sales Operations Analyst
JM Eagle

About the job
SUMMARY
Reporting daily to the company headquarters in Los Angeles, the Sales Operations Analyst role is will be responsible for processing multiple purchase orders (POs), coordinating deliveries, managing product sourcing, and ensuring seamless operational processes. This role requires exceptional detail orientation, optimal organization, strong technical skills, outstanding customer service abilities, and a commitment to accuracy in a fast-paced environment.

WORK LOCATION
Onsite. 5200 W. Century Blvd. Los Angeles, Ca. 90045

ESSENTIAL DUTIES AND RESPONSIBILITIES include the following. Other duties may be assigned.
Purchase Order Management: Process multiple purchase orders accurately and efficiently, adhering to company policies and timelines, and ensuring the highest level of service possible under the guidan

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [37]:
# ── Strategy 1 ──

# Strategy 1: Larger recursive chunks
# This keeps more surrounding context together, but may mix multiple JD sections in one chunk.

strategy_1_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_documents = jd_documents + resume_documents

strategy_1_chunks = strategy_1_splitter.split_documents(all_documents)

print("Strategy 1 complete.")
print("Total documents before chunking:", len(all_documents))
print("Total chunks created:", len(strategy_1_chunks))

# Count chunks by document type
strategy_1_counts = {}

for chunk in strategy_1_chunks:
    doc_type = chunk.metadata.get("doc_type", "unknown")
    
    if doc_type not in strategy_1_counts:
        strategy_1_counts[doc_type] = 0
    
    strategy_1_counts[doc_type] += 1

print("Chunks by document type:")
print(strategy_1_counts)

# Preview one chunk
print("\nSample chunk preview:")
print(strategy_1_chunks[0].page_content[:1000])

Strategy 1 complete.
Total documents before chunking: 11
Total chunks created: 83
Chunks by document type:
{'job_description': 77, 'resume': 6}

Sample chunk preview:
Sales Operations Analyst
JM Eagle

About the job
SUMMARY
Reporting daily to the company headquarters in Los Angeles, the Sales Operations Analyst role is will be responsible for processing multiple purchase orders (POs), coordinating deliveries, managing product sourcing, and ensuring seamless operational processes. This role requires exceptional detail orientation, optimal organization, strong technical skills, outstanding customer service abilities, and a commitment to accuracy in a fast-paced environment.

WORK LOCATION
Onsite. 5200 W. Century Blvd. Los Angeles, Ca. 90045


In [39]:
# ── Strategy 2 ──

# Strategy 2: Smaller section-aware recursive chunks
# This is designed for job descriptions because JDs often have sections like:
# Responsibilities, Qualifications, Requirements, Skills, and Preferred Qualifications.

strategy_2_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=[
        "\n\nResponsibilities",
        "\n\nQualifications",
        "\n\nRequirements",
        "\n\nPreferred Qualifications",
        "\n\nSkills",
        "\n\nAbout",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

strategy_2_chunks = strategy_2_splitter.split_documents(all_documents)

print("Strategy 2 complete.")
print("Total documents before chunking:", len(all_documents))
print("Total chunks created:", len(strategy_2_chunks))

# Count chunks by document type
strategy_2_counts = {}

for chunk in strategy_2_chunks:
    doc_type = chunk.metadata.get("doc_type", "unknown")
    
    if doc_type not in strategy_2_counts:
        strategy_2_counts[doc_type] = 0
    
    strategy_2_counts[doc_type] += 1

print("Chunks by document type:")
print(strategy_2_counts)

# Preview one chunk
print("\nSample chunk preview:")
print(strategy_2_chunks[0].page_content[:1000])

Strategy 2 complete.
Total documents before chunking: 11
Total chunks created: 139
Chunks by document type:
{'job_description': 131, 'resume': 8}

Sample chunk preview:
Sales Operations Analyst
JM Eagle


In [43]:
# ── Compare strategies ──

def summarize_chunks(chunks, strategy_name):
    chunk_lengths = [len(chunk.page_content) for chunk in chunks]
    
    summary = {
        "strategy": strategy_name,
        "total_chunks": len(chunks),
        "avg_chunk_length": round(sum(chunk_lengths) / len(chunk_lengths), 2),
        "min_chunk_length": min(chunk_lengths),
        "max_chunk_length": max(chunk_lengths),
        "jd_chunks": sum(1 for chunk in chunks if chunk.metadata.get("doc_type") == "job_description"),
        "resume_chunks": sum(1 for chunk in chunks if chunk.metadata.get("doc_type") == "resume")
    }
    
    return summary

chunking_comparison = pd.DataFrame([
    summarize_chunks(strategy_1_chunks, "Strategy 1: Larger recursive chunks"),
    summarize_chunks(strategy_2_chunks, "Strategy 2: Smaller section-aware chunks")
])

display(chunking_comparison)

,strategy,total_chunks,avg_chunk_length,min_chunk_length,max_chunk_length,jd_chunks,resume_chunks
0,Strategy 1: Larger recursive chunks,83,720.89,54,990,77,6
1,Strategy 2: Smaller section-aware chunks,139,421.27,14,600,131,8


### Chunking Decision

**Which strategy did you choose?**  I selected Strategy 2 as the final chunking approach. 

**Why?**  Streagy 2 used smaller, section-aware recursive chunks with a chunk size of 600 and an overlap of 100. This strategy is better suited for job descriptions because many postings are organized into sections such as responsibilities, qualifications, requirements, preferred qualifications, and skills. Smaller chunks make retrieval more focused and easier to compare with evidence from the resume. Precise skill gaps, keyword alignment, and fit summary analysis are all stronger with improved and more specific chunking. 

**Final settings (chunk_size, overlap):** Smaller section-aware chunks, max_chunk_length = 600, overlap = 100

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
# ── Create embeddings and vector store ──


In [ ]:
# ── Verify: run a test similarity search ──


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [ ]:
# ── Initialize LLM ──


In [ ]:
# ── Analysis 1: Skill Gap Report ──


In [ ]:
# ── Analysis 2: Keyword Alignment ──


In [ ]:
# ── Analysis 3: Fit Summary ──


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot version of your chosen analysis ──


In [ ]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [ ]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*